In [3]:
import os
import re
from collections import Counter

import soundfile as sf

# ---- Audio paths: the folder that CONTAINS the wavs/ folder ----
PATHNIRWANA_AUDIO = r"D:\RUSL\Final Project\Dataset\Pathnirwana"
SPEAKER_01_AUDIO  = r"D:\RUSL\Final Project\Dataset\Hexacoder"
SPEAKER_02_AUDIO  = r"D:\RUSL\Final Project\Dataset\Review Master"

# ---- Metadata paths ----
TRAIN_DIR = r"D:\RUSL\Final Project\TTS\voicelk_ml\data\train"
OUT_DIR = r"D:\RUSL\Final Project\TTS\voicelk_ml\data\train_ready"

# metadata file -> audio folder
DATASETS = {
    "pathnirwana_metadata_final.csv": PATHNIRWANA_AUDIO,
    "speaker_01_metadata_final.csv": SPEAKER_01_AUDIO,
    "speaker_02_metadata_final.csv": SPEAKER_02_AUDIO,
}

CHECK_AUDIO = True               # False: skip wav checks (no audio on this machine)
MIN_DUR, MAX_DUR = 1.0, 20.0     # seconds
MIN_RATE, MAX_RATE = 4.0, 25.0   # IPA symbols (no spaces/punctuation) per second
MIN_SYMBOLS = 2                  # drops rows like "." or "a"
PUNCT = set(".,?!'\" ")

os.makedirs(OUT_DIR, exist_ok=True)

# Stop early if an audio folder is wrong, instead of writing empty files
if CHECK_AUDIO:
    for meta_name, audio_root in DATASETS.items():
        wav_dir = os.path.join(audio_root, "wavs")
        if not os.path.isdir(wav_dir):
            raise SystemExit(f"{meta_name}: no wavs folder at {wav_dir} - fix its audio path")
        print(f"{meta_name}: {len(os.listdir(wav_dir))} files in {wav_dir}")


def clean_ipa(ipa: str) -> str:
    ipa = ipa.strip()
    # Undo CSV quoting: "...""hello""..." -> ..."hello"...
    if len(ipa) >= 2 and ipa.startswith('"') and ipa.endswith('"'):
        ipa = ipa[1:-1].replace('""', '"')
    return re.sub(r"\s+", " ", ipa).strip()


all_rows = []

for meta_name, audio_root in DATASETS.items():
    reasons = Counter()
    kept = []

    with open(os.path.join(TRAIN_DIR, meta_name), "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for line in lines:
        parts = line.split("|", 2)          # path | speaker | ipa
        if len(parts) != 3:
            reasons["bad line"] += 1
            continue
        path, speaker, ipa = parts[0].strip(), parts[1].strip(), clean_ipa(parts[2])

        symbols = sum(1 for c in ipa if c not in PUNCT)
        if symbols < MIN_SYMBOLS:
            reasons["empty / too short text"] += 1
            continue
        if "|" in ipa:
            reasons["'|' inside ipa"] += 1
            continue

        if CHECK_AUDIO:
            wav = os.path.join(audio_root, path)
            if not os.path.exists(wav):
                reasons["wav missing"] += 1
                continue
            dur = sf.info(wav).duration
            if dur < MIN_DUR:
                reasons[f"shorter than {MIN_DUR}s"] += 1
                continue
            if dur > MAX_DUR:
                reasons[f"longer than {MAX_DUR}s"] += 1
                continue
            rate = symbols / dur
            if rate < MIN_RATE or rate > MAX_RATE:
                reasons["text/audio length mismatch"] += 1
                continue

        kept.append((path, ipa, speaker))

    stem = meta_name.replace("_metadata_final.csv", "")
    with open(os.path.join(OUT_DIR, f"{stem}_single.txt"), "w", encoding="utf-8", newline="\n") as f:
        for path, ipa, _ in kept:
            f.write(f"{path}|{ipa}\n")

    all_rows.extend(kept)
    print(f"{meta_name}: {len(lines)} -> {len(kept)} kept")
    for reason, n in reasons.most_common():
        print(f"    removed {n:4d}  {reason}")

with open(os.path.join(OUT_DIR, "all_speakers.txt"), "w", encoding="utf-8", newline="\n") as f:
    for path, ipa, speaker in all_rows:
        f.write(f"{path}|{ipa}|{speaker}\n")

print(f"\nall_speakers.txt: {len(all_rows)} rows, speakers {sorted(Counter(r[2] for r in all_rows).items())}")


pathnirwana_metadata_final.csv: 3300 files in D:\RUSL\Final Project\Dataset\Pathnirwana\wavs
speaker_01_metadata_final.csv: 4399 files in D:\RUSL\Final Project\Dataset\Hexacoder\wavs
speaker_02_metadata_final.csv: 3196 files in D:\RUSL\Final Project\Dataset\Review Master\wavs
pathnirwana_metadata_final.csv: 3300 -> 3283 kept
    removed   16  longer than 20.0s
    removed    1  shorter than 1.0s
speaker_01_metadata_final.csv: 4139 -> 3979 kept
    removed   90  shorter than 1.0s
    removed   66  text/audio length mismatch
    removed    4  empty / too short text
speaker_02_metadata_final.csv: 3116 -> 3087 kept
    removed   15  shorter than 1.0s
    removed   14  text/audio length mismatch

all_speakers.txt: 10349 rows, speakers [('0', 3283), ('1', 3979), ('2', 3087)]
